<a href="https://colab.research.google.com/github/DJ-Ortega/unsupervised-trading-strat/blob/main/Unsupervised_strat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unsupervised Learning Trading Strategy

* Download/Load SP500 stocks prices data.
* Calculate different features and indicators on each stock.
* Aggregate on monthly level and filter top 150 most liquid stocks.
* Calculate Monthly Returns for different time-horizons.
* Download Fama-French Factors and Calculate Rolling Factor Betas.
* For each month fit a K-Means Clustering Algorithm to group similar assets based on their features.
* For each month select assets based on the cluster and form a portfolio based on Efficient Frontier max sharpe ratio optimization.
* Visualize Portfolio returns and compare to SP500 returns.

Be mindful that this backtest does not take in survivorship bias

In [5]:
!pip install pandas-datareader --quiet
!pip install pyportfolioopt --quiet
!pip install pandas_ta --quiet
!pip install --upgrade yfinance --quiet
!pip install pandas==2.3.3 --quiet

In [1]:
from statsmodels.regression.rolling import RollingOLS
import pandas_datareader.data as web
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pandas as pd
import numpy as np
import datetime as dt
import yfinance as yf
import pandas_ta
import warnings
warnings.filterwarnings('ignore')

## 1. Download/Load SP500 stocks prices data.

In [28]:
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'Mozilla/5.0 (student learning project)'}
response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]
sp500['Symbol'] = sp500['Symbol'].str.replace('.', '-')
symbols_list = sp500['Symbol'].unique().tolist()

end_date = '2026-10-30'
start_date = pd.to_datetime(end_date) - pd.DateOffset(365*8)

raw_df = yf.download(tickers=symbols_list,
                     start=start_date,
                     end=end_date)

[*********************100%***********************]  503 of 503 completed


In [31]:
#cleaning the dataframe a little bit
df = raw_df.stack()
df.index.names = ['date', 'ticker']
df.columns = df.columns.str.lower()
df

Price                   close        high         low        open       volume
date       ticker                                                             
2018-11-01 A        61.491657   61.604793   60.888244   61.350231    1882400.0
           AAPL     52.496872   52.529945   51.218822   51.747997  233292800.0
           ABBV     57.475533   57.790822   55.533647   55.684125   10671700.0
           ABT      60.562416   60.735079   59.345121   59.647285    6250800.0
           ACGL     27.281279   27.576057   26.739267   27.119626    2081000.0
...                       ...         ...         ...         ...          ...
2026-10-29 XYZ      76.949997   77.559998   74.809998   74.830002    4331600.0
           YUM     154.550003  154.750000  152.000000  152.520004    1606700.0
           ZBH      97.470001   97.989998   96.260002   97.330002    2040800.0
           ZBRA    377.380005  389.230011  374.549988  384.929993     527500.0
           ZTS      78.270004   78.746772   77.236998   77.912425    2935500.0

[987105 rows x 5 columns]

## 2. Calculate features and technical indicators for each stock.

* Garman-Klass Volatility
* RSI
* Bollinger Bands
* ATR
* MACD
* Dollar Volume

\begin{equation}
\text{Garman-Klass Volatility} = \frac{(\ln(\text{High}) - \ln(\text{Low}))^2}{2} - (2\ln(2) - 1)(\ln(\text{Adj Close}) - \ln(\text{Open}))^2
\end{equation}